# Convoluted Neural Networks
- Convoluted Neural Networks (aka CNNs), are a special type of neural network, that are heavily used and applied in Computer Vision.
- ANNs are bad for image tasks. Imagine a 224 x 224 image (low quality square) with 3 channels for colour (RGB). That's 150,528 pixels, so 150,528 inputs. A connected layer with 1000 neurons would need 150 million weights. 
- That's very inefficient, and doesn't capture relationships between adjacent pixels. For example, edges and shapes that make up an entire image.
- This is what CNNs were built to address.

### How CNNs work
- They have a sliding type of behaviour. They use a filter (aka a kernel). This filter is a shape that slides over an image and is just a tiny matrix of numbers (e.g. 3x3).
- Let's reduce dimensions for simplicity. Imagine a 5x5 image and a 3x3 filter. You can slide it across a 5x5 image/input a total of 9 times. A visualization:


![CNN filter visualisation](../media/CNNs%20filter.png)


- Each filter has a feature/pattern that it "looks" for, and will be populated with values accordingly. For example, a CNN looking for a straight horizontal line would look like:

<table>
  <tr>
    <td>0</td><td>0</td><td>0</td>
  </tr>
  <tr>
    <td>1</td><td>1</td><td>1</td>
  </tr>
  <tr>
    <td>0</td><td>0</td><td>0</td>
  </tr>
</table>

- Then, for each position (for example, the 9 positions we mentioned earlier), the network performs a dot product between the filter and the current patch of the input we are on. This gives 9 numbers and therefore a 3x3 matrix, also known as a feature map.
- This feature map highlights where the filter’s pattern appears.

- This can now be applied to create a visualization of what an actual convoluted neural network looks like. In our case, we would have 25 input nodes, 9 nodes in the convoluted layer, and 1 node in the output. (how to go from hidden/convoluted layer to output is covered later).

- But how does the model learn what pattern(s) should filters look for? To understand this, we must first learn how pooling works. (A knowledge of activation functions is also required, but I already cover them in my neural_networks_fundamentals notebook). 

- Now let's go over pooling.

#### Pooling
- Process the makes feature maps smaller while keeping the strongest/most important activations. (No weights or nothing to learn like that, its a hard coded rule that gets applied across regions).
- Types:
  - Max pooling: takes the largest value in the region
  - Average pooling: looks for average value in the region
  - CNNs usually use max pooling, because if a filter was picked up somewhere, we care that that feature took place at all, not exactly how strong it took at all places.


Let's look at a concrete example of max pooling. Suppose we have the following 4x4 feature map:

<table>
  <tr>
    <td>1</td><td>3</td><td>2</td><td>4</td>
  </tr>
  <tr>
    <td>5</td><td>6</td><td>7</td><td>8</td>
  </tr>
  <tr>
    <td>9</td><td>2</td><td>0</td><td>3</td>
  </tr>
  <tr>
    <td>4</td><td>7</td><td>5</td><td>1</td>
  </tr>
</table>

If we apply a 2x2 max pooling operation with a stride of 2, we divide the feature map into four 2x2 regions:

- Top-left: <table style="display:inline"><tr><td>1</td><td>3</td></tr><tr><td>5</td><td>6</td></tr></table>
- Top-right: <table style="display:inline"><tr><td>2</td><td>4</td></tr><tr><td>7</td><td>8</td></tr></table>
- Bottom-left: <table style="display:inline"><tr><td>9</td><td>2</td></tr><tr><td>4</td><td>7</td></tr></table>
- Bottom-right: <table style="display:inline"><tr><td>0</td><td>3</td></tr><tr><td>5</td><td>1</td></tr></table>

For each region, we take the maximum value:

- Top-left: max(1, 3, 5, 6) = 6
- Top-right: max(2, 4, 7, 8) = 8
- Bottom-left: max(9, 2, 4, 7) = 9
- Bottom-right: max(0, 3, 5, 1) = 5

The resulting pooled feature map is:

<table>
  <tr>
    <td>6</td><td>8</td>
  </tr>
  <tr>
    <td>9</td><td>5</td>
  </tr>
</table>

This operation reduces the size of the feature map while keeping the most prominent values from each region, helping the network focus on the most important features.


## Stride
- Very simple. Filters slide over input images/feature maps one pixel at a time, so that is stride = 1.
- If stride = 2, the filter jumps 2 pixels at a time.
- This causes the feature map to lose some pixels if you would have had a smaller stride, as pixels would get jumped over/skipped. This leads to a "zooming out" effect, and makes the filter map smaller.
- The point is to sacrifice a little bit of accuracy/performance for faster speed.

## Padding
- Filters shrink image sizes.
- Padding just adds 0s around the border of the input image, so that the size of the input image stays the same after the filter is completely applied to the whole image.
- Purpose: keeps edge information preserved, as edges otherwise just disappear, and you lose out on potentially important information.
- For example, here is a 5x5 matrix with values from 1 to 25:

<table>
  <tr>
    <td>1</td><td>2</td><td>3</td><td>4</td><td>5</td>
  </tr>
  <tr>
    <td>6</td><td>7</td><td>8</td><td>9</td><td>10</td>
  </tr>
  <tr>
    <td>11</td><td>12</td><td>13</td><td>14</td><td>15</td>
  </tr>
  <tr>
    <td>16</td><td>17</td><td>18</td><td>19</td><td>20</td>
  </tr>
  <tr>
    <td>21</td><td>22</td><td>23</td><td>24</td><td>25</td>
  </tr>
</table>

Now imagine you use a 3x3 filter with stride = 1. Now here's what happens:
  - The filter can only sit in the middle positions horizontally and vertically. (Meaning, the filter can only sit with 7, 8, 9, 12, ..., 18, 19 at the center). 
  - So pixels 1, 2, 4, 5, 21, 22, 24, 25 (corner pixels) are never at the center of any filter window.
  - They never have full proper effect on the outputs from this current image. 
  - This can be problematic. Imagine detecting a face in an image. What if the eyes are near the border?
    - Without padding, those features could get ignored (lost as you go deeper).
    - With padding, the filter can still process patterns that touch the boundary.

### Activation Functions in CNNs
- After a feature map is generated, an activation function can be applied to element-wise, before it used to be applied into another feature map.
- Without it, stacking multiple convolutions is useless, the whole thing would just become one massive linear filter actually.
- With activation functions, CNN can now build layers of abstraction:
  - First conv + ReLU: detect edges.
  - Next conv + ReLU: combine edges into shapes.
  - Deeper conv + ReLU: combine shapes into objects.

- They also include biases, which also get applied element-wise.

### Flattening
- What now happens is, all feature maps get "flattened" so that they can be expressed and interacted with as if they were 1 single dense linear layer, which can then be followed by more dense layers to segue into a neural network type of architecture, which is something we are already extremely familiar with.
- So lets say that your convoluted layer(s) produced 8 feature maps, each of size 4x4:
  - Each feature map is 4x4 = 16 numbers.
  - Feature maps combined are 8 x 16 = 128 numbers.
- These 128 numbers can be flattened into 1 long vector [$f_{1,1​},f_{1,2}​,…,f_{1,16}​,f_{2,1​},f_{2,2}​,…,f_{i,j}...,f_{8,16}​$]
  - Where $i$ is the $i^{th}$ feature map produced, and $j$ is the $j^{th}$ number of that feature map.

Here's a visualization of what the architecture looks like:

![CNN Visualization](../media/cnn_visualization.webp)

### Dropout
- You may have noticed "dropout" in the screenshot above. It's a very simple concept.
- During training, what happens is that some nodes in the network are randomly "dropped" (having their output set to 0).
- This prevents the network from becoming over reliant on specific nodes, which prevents overfitting (when the model becomes specifically good at the dataset it was trained on, while underperforming on actual data/tasks).
- Example: if dropout = 0.5, half the neurons are turned off each step (chosen randomly).
- Then, at actual test time, nothing is dropped and all weights are used, but weights are scaled.
- The weights are scaled at test time to ensure the outputs remain consistent between training and testing. This compensates for the fact that, during training, only a subset of neurons were active at each step.
- Example:
  - Suppose you have a layer with 4 neurons, and you use dropout with a rate of 0.5 during training. On each training step, about half of the neurons are randomly "dropped out" (set to zero). For instance, if the outputs of the neurons before dropout are [2, 4, 6, 8], after applying dropout (randomly dropping the 2nd and 4th neurons), you might get [2, 0, 6, 0].
  - At test time, all neurons are active, but to keep the expected output the same as during training, each neuron's output is multiplied by the dropout rate (in this case, 0.5). So, the output becomes [1, 2, 3, 4] instead of [2, 4, 6, 8]. This scaling ensures that the network's behavior is consistent between training and testing.

### GAP (Global Average Pooling)
- At the flattening step, instead of flattening all final feature maps into 1 single one-dimensional huge vector:
  - We just take the average value of each feature map.
  - So if you have 256 feature maps of size 7x7 each, we would take the average value in each of those 256 feature maps, and put the 256 numbers into a single vector, of length 256 (1 number per feature map), which is then treated as a dense linear layer, instead of unrolling a massive 1D vector of 256 x 7 x 7 elements after the flattening step.
- This gives us a much smaller input vector to pass into the dense layer, and avoids overfitting. (It avoids overfitting because fewer parameters -> less chance of memorizing noise -> better generalization.)
  - GAP sort of forces the model to think "Does this feature REALLY exist in the map at all?", instead of "where exactly is it and how strong is it at each pixel?". It focuses more on the feature's actual presence, instead of the exact position of the feature(s).

### Architectures
- There are many architectures that come with CNNs. Some prevalent ones:
- LeNet (1998): first CNN.
- AlexNet (2012).
- ResNet (2015): it has skip connections, and has super deep networks which are trainable.